In [62]:
# Library for handling dataset
import pandas as pd
import numpy as np

# Library for visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Libraries for machine learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score

# Logistic Regression model
from sklearn.linear_model import LogisticRegression

# Decision Tree model
from sklearn.tree import DecisionTreeClassifier

# Hyperparameter tuning
from sklearn.model_selection import GridSearchCV

In [103]:
# Importing zipfile library
import zipfile

# Opening ZIP file
with zipfile.ZipFile('credit_score.zip') as z:
    
    # Reading the correct CSV file
    with z.open('credit_score.csv') as f:
        df = pd.read_csv(f, low_memory=False)

# Displaying dataset
print(df.head())

# Checking dataset shape
print(df.shape)

       ID Customer_ID     Month           Name   Age          SSN Occupation  \
0  0x1602   CUS_0xd40   January  Aaron Maashoh    23  821-00-0265  Scientist   
1  0x1603   CUS_0xd40  February  Aaron Maashoh    23  821-00-0265  Scientist   
2  0x1604   CUS_0xd40     March  Aaron Maashoh  -500  821-00-0265  Scientist   
3  0x1605   CUS_0xd40     April  Aaron Maashoh    23  821-00-0265  Scientist   
4  0x1606   CUS_0xd40       May  Aaron Maashoh    23  821-00-0265  Scientist   

  Annual_Income  Monthly_Inhand_Salary  Num_Bank_Accounts  ...  Credit_Mix  \
0      19114.12            1824.843333                  3  ...           _   
1      19114.12                    NaN                  3  ...        Good   
2      19114.12                    NaN                  3  ...        Good   
3      19114.12                    NaN                  3  ...        Good   
4      19114.12            1824.843333                  3  ...        Good   

   Outstanding_Debt Credit_Utilization_Ratio     C

In [105]:
# Displaying column names and datatypes
print(df.info())

# Statistical summary of numerical columns
print(df.describe())

# Checking null values
print(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 28 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   ID                        100000 non-null  object 
 1   Customer_ID               100000 non-null  object 
 2   Month                     100000 non-null  object 
 3   Name                      90015 non-null   object 
 4   Age                       100000 non-null  object 
 5   SSN                       100000 non-null  object 
 6   Occupation                100000 non-null  object 
 7   Annual_Income             100000 non-null  object 
 8   Monthly_Inhand_Salary     84998 non-null   float64
 9   Num_Bank_Accounts         100000 non-null  int64  
 10  Num_Credit_Card           100000 non-null  int64  
 11  Interest_Rate             100000 non-null  int64  
 12  Num_of_Loan               100000 non-null  object 
 13  Type_of_Loan              88592 non-null   ob

In [107]:
# Dropping columns not useful for prediction
df = df.drop(['ID', 'Customer_ID', 'Name', 'SSN'], axis=1)

# Checking remaining columns
print(df.columns)

Index(['Month', 'Age', 'Occupation', 'Annual_Income', 'Monthly_Inhand_Salary',
       'Num_Bank_Accounts', 'Num_Credit_Card', 'Interest_Rate', 'Num_of_Loan',
       'Type_of_Loan', 'Delay_from_due_date', 'Num_of_Delayed_Payment',
       'Changed_Credit_Limit', 'Num_Credit_Inquiries', 'Credit_Mix',
       'Outstanding_Debt', 'Credit_Utilization_Ratio', 'Credit_History_Age',
       'Payment_of_Min_Amount', 'Total_EMI_per_month',
       'Amount_invested_monthly', 'Payment_Behaviour', 'Monthly_Balance',
       'Credit_Score'],
      dtype='object')


In [109]:
# Replacing empty spaces with NaN values
df.replace(r'^\s*$', np.nan, regex=True, inplace=True)

# Filling missing values using forward fill
df.ffill(inplace=True)

# Checking missing values again
print(df.isnull().sum())

Month                       0
Age                         0
Occupation                  0
Annual_Income               0
Monthly_Inhand_Salary       0
Num_Bank_Accounts           0
Num_Credit_Card             0
Interest_Rate               0
Num_of_Loan                 0
Type_of_Loan                0
Delay_from_due_date         0
Num_of_Delayed_Payment      0
Changed_Credit_Limit        0
Num_Credit_Inquiries        0
Credit_Mix                  0
Outstanding_Debt            0
Credit_Utilization_Ratio    0
Credit_History_Age          0
Payment_of_Min_Amount       0
Total_EMI_per_month         0
Amount_invested_monthly     0
Payment_Behaviour           0
Monthly_Balance             0
Credit_Score                0
dtype: int64


In [110]:
# Creating LabelEncoder object
le = LabelEncoder()

# Encoding categorical columns
for col in df.select_dtypes(include='object').columns:
    df[col] = le.fit_transform(df[col].astype(str))

# Displaying dataset after encoding
print(df.head())

   Month  Age  Occupation  Annual_Income  Monthly_Inhand_Salary  \
0      3  308          12           6011            1824.843333   
1      2  308          12           6011            1824.843333   
2      6    0          12           6011            1824.843333   
3      0  308          12           6011            1824.843333   
4      7  308          12           6011            1824.843333   

   Num_Bank_Accounts  Num_Credit_Card  Interest_Rate  Num_of_Loan  \
0                  3                4              3          244   
1                  3                4              3          244   
2                  3                4              3          244   
3                  3                4              3          244   
4                  3                4              3          244   

   Type_of_Loan  ...  Credit_Mix  Outstanding_Debt  Credit_Utilization_Ratio  \
0           128  ...           3             12062                 26.822620   
1           128  ...  

In [111]:
# Input features
X = df.drop('Credit_Score', axis=1)

# Target variable
y = df['Credit_Score']

# Displaying shapes
print(X.shape)
print(y.shape)


(100000, 23)
(100000,)


In [113]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# Checking training and testing shapes
print(X_train.shape)
print(X_test.shape)

(80000, 23)
(20000, 23)


In [114]:
# Creating scaler object
scaler = StandardScaler()

# Scaling training data
X_train_scaled = scaler.fit_transform(X_train)

# Scaling testing data
X_test_scaled = scaler.transform(X_test)


In [115]:
# Creating Logistic Regression model
lr = LogisticRegression(max_iter=5000)

# Training the model
lr.fit(X_train_scaled, y_train)

# Predicting test data
y_pred_lr = lr.predict(X_test_scaled)

# Calculating accuracy
lr_accuracy = accuracy_score(y_test, y_pred_lr)

# Printing accuracy
print("Logistic Regression Accuracy:", lr_accuracy)


Logistic Regression Accuracy: 0.5925


In [119]:

# Creating Decision Tree model
dt = DecisionTreeClassifier()

# Training model
dt.fit(X_train, y_train)

# Predicting test data
y_pred_dt = dt.predict(X_test)

# Calculating accuracy
dt_accuracy = accuracy_score(y_test, y_pred_dt)

# Printing accuracy
print("Decision Tree Accuracy:", dt_accuracy)



Decision Tree Accuracy: 0.67925


In [121]:
# Parameter grid
params = {
    'max_depth': [5, 10, 15],
    'criterion': ['gini', 'entropy']
}

# Applying GridSearchCV
grid = GridSearchCV(
    DecisionTreeClassifier(),
    params,
    cv=5
)

# Training GridSearchCV
grid.fit(X_train, y_train)

# Best parameters
print("Best Parameters:", grid.best_params_)

# Best model
best_model = grid.best_estimator_

# Predicting using best model
y_pred_best = best_model.predict(X_test)

# Accuracy after tuning
best_accuracy = accuracy_score(y_test, y_pred_best)

# Printing tuned accuracy
print("Tuned Decision Tree Accuracy:", best_accuracy)


Best Parameters: {'criterion': 'entropy', 'max_depth': 10}
Tuned Decision Tree Accuracy: 0.699


In [123]:
# Importing Random Forest
from sklearn.ensemble import RandomForestClassifier

# Creating model
rf = RandomForestClassifier(n_estimators=100)

# Training model
rf.fit(X_train, y_train)

# Predicting test data
y_pred_rf = rf.predict(X_test)

# Calculating accuracy
rf_accuracy = accuracy_score(y_test, y_pred_rf)

# Printing accuracy
print("Random Forest Accuracy:", rf_accuracy)


Random Forest Accuracy: 0.78365


In [124]:

print("\n===== MODEL ACCURACY COMPARISON =====")

print("Logistic Regression Accuracy :", lr_accuracy)

print("Decision Tree Accuracy :", dt_accuracy)

print("Tuned Decision Tree Accuracy :", best_accuracy)

print("Random Forest Accuracy :", rf_accuracy)


===== MODEL ACCURACY COMPARISON =====
Logistic Regression Accuracy : 0.5925
Decision Tree Accuracy : 0.67925
Tuned Decision Tree Accuracy : 0.699
Random Forest Accuracy : 0.78365
